# Lab 03: Deploy CertAgent to Amazon Bedrock AgentCore Runtime

## Overview

Deploy a persistent AI agent to AgentCore Runtime using the `bedrock-agentcore-starter-toolkit`.

```
User --> AgentCore Runtime (CertAgent) --> Lambda tools --> DynamoDB/Secrets Manager
```

The agent uses Strands Agents SDK with `BedrockAgentCoreApp` which handles the HTTP protocol automatically.

**Estimated time:** 30 minutes

**Important:** Run all cells in order. Do NOT restart kernel between steps.

## Step 1 - Install dependencies

In [ ]:
!pip install -q bedrock-agentcore-starter-toolkit strands-agents strands-agents-bedrock bedrock-agentcore boto3

## Step 2 - Load workshop configuration

In [ ]:
import json, pathlib, boto3, os
from boto3.session import Session

config = json.loads(pathlib.Path('/tmp/certagent_config.json').read_text())
globals().update(config)
REPO_DIR = pathlib.Path(REPO_DIR)

boto_session = Session(region_name=AWS_REGION)
region = AWS_REGION

print(f'Region: {AWS_REGION}')
print(f'Scan Lambda: {LAMBDA_SCAN}')
print('Config loaded')

## Step 3 - Create a clean working directory

The toolkit zips everything in the current directory. We work from a clean folder.

In [ ]:
import os, shutil, glob

WORK_DIR = '/home/sagemaker-user/certagent-deploy'
if os.path.exists(WORK_DIR):
    shutil.rmtree(WORK_DIR)
os.makedirs(WORK_DIR)
os.chdir(WORK_DIR)
print(f'Working directory: {os.getcwd()}')

## Step 4 - Write the CertAgent server code

Key elements:
- `BedrockAgentCoreApp()` handles HTTP protocol (`/invocations`, `/ping`)
- `@app.entrypoint` decorator marks the invocation function
- `app.run()` starts the server on port 8080

In [ ]:
%%writefile certagent_server.py
import os
import json
import boto3
from strands import Agent, tool
from strands.models import BedrockModel
from bedrock_agentcore.runtime import BedrockAgentCoreApp

app = BedrockAgentCoreApp()

AWS_REGION = os.environ.get('AWS_REGION', 'us-east-1')
lambda_client = boto3.client('lambda', region_name=AWS_REGION)

LAMBDA_SCAN = os.environ.get('LAMBDA_SCAN', 'certagent-scan-certificates')
LAMBDA_RENEW = os.environ.get('LAMBDA_RENEW', 'certagent-renew-certificate')
LAMBDA_INVENTORY = os.environ.get('LAMBDA_INVENTORY', 'certagent-list-inventory')

def invoke_lambda(fn_name, payload):
    r = lambda_client.invoke(FunctionName=fn_name, InvocationType='RequestResponse',
                             Payload=json.dumps(payload))
    raw = json.loads(r['Payload'].read())
    return raw.get('body', raw)

@tool
def scan_certificates(threshold_days: int = 30, use_mock: bool = True) -> str:
    """Scan for certificates expiring within the given threshold days.
    Args:
        threshold_days: Number of days to look ahead.
        use_mock: Use mock data instead of real DigiCert API.
    """
    result = invoke_lambda(LAMBDA_SCAN, {'threshold_days': threshold_days, 'use_mock': use_mock})
    return json.dumps(result, indent=2, default=str)

@tool
def renew_certificate(order_id: str, common_name: str, use_mock: bool = True) -> str:
    """Renew a certificate by order ID and domain name.
    Args:
        order_id: The DigiCert order ID to renew.
        common_name: The domain name of the certificate.
        use_mock: Use mock mode.
    """
    result = invoke_lambda(LAMBDA_RENEW, {
        'order_id': order_id, 'common_name': common_name,
        'sans': [common_name], 'use_mock': use_mock
    })
    return json.dumps(result, indent=2, default=str)

@tool
def list_inventory(status: str = 'all') -> str:
    """List the certificate inventory.
    Args:
        status: Filter: all, pending, submitted, issued, completed.
    """
    result = invoke_lambda(LAMBDA_INVENTORY, {'status': status})
    return json.dumps(result, indent=2, default=str)

model = BedrockModel(model_id='us.anthropic.claude-sonnet-4-5-20250929-v1:0', region_name=AWS_REGION)
agent = Agent(
    model=model,
    tools=[scan_certificates, renew_certificate, list_inventory],
    system_prompt='You are CertAgent, an AI ops agent for TLS/SSL certificate lifecycle. Use mock mode. Be concise.'
)

@app.entrypoint
def invoke_certagent(payload):
    """Invoke the agent with a payload from AgentCore Runtime."""
    user_input = payload.get('prompt', '')
    print(f'User input: {user_input}')
    response = agent(user_input)
    return response.message['content'][0]['text']

if __name__ == '__main__':
    app.run()

## Step 5 - Write requirements file

In [ ]:
%%writefile requirements.txt
strands-agents
strands-agents-bedrock
bedrock-agentcore
boto3>=1.35.0

## Step 6 - Configure AgentCore Runtime

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime

agentcore_runtime = Runtime()
agent_name = 'certagent'

print('Configuring...')
response = agentcore_runtime.configure(
    entrypoint='certagent_server.py',
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file='requirements.txt',
    region=region,
    agent_name=agent_name,
)
print('Configuration completed')

## Step 7 - Launch to AgentCore Runtime

This builds the container via CodeBuild (ARM64), pushes to ECR, and deploys. Takes 3-5 minutes.

In [ ]:
print('Launching CertAgent to AgentCore Runtime...')
print('This takes 3-5 minutes...')
launch_result = agentcore_runtime.launch()
print(f'\nAgent ARN: {launch_result.agent_arn}')
print(f'Agent ID: {launch_result.agent_id}')
print('Deployment successful!')

## Step 8 - Grant Lambda invoke permission to the agent runtime role

The auto-created execution role needs permission to invoke our Lambda functions.

In [ ]:
import boto3, json

iam_client = boto3.client('iam')
sts_client = boto3.client('sts')
account_id = sts_client.get_caller_identity()['Account']

# Find the execution role created by the toolkit
roles = iam_client.list_roles(MaxItems=200)['Roles']
runtime_roles = [r for r in roles if 'BedrockAgentCoreSDKRuntime' in r['RoleName']]
runtime_roles.sort(key=lambda r: r['CreateDate'], reverse=True)

if runtime_roles:
    role_name = runtime_roles[0]['RoleName']
    print(f'Adding Lambda invoke permission to: {role_name}')
    iam_client.put_role_policy(
        RoleName=role_name,
        PolicyName='LambdaInvoke',
        PolicyDocument=json.dumps({
            'Version': '2012-10-17',
            'Statement': [{
                'Effect': 'Allow',
                'Action': 'lambda:InvokeFunction',
                'Resource': f'arn:aws:lambda:{AWS_REGION}:{account_id}:function:certagent-*'
            }]
        })
    )
    print('Permission granted')
else:
    print('No AgentCore runtime role found')

## Step 9 - Wait for runtime to be READY

In [ ]:
import time

status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f'Status: {status}')
print(f'\nFinal status: {status}')

## Step 10 - Invoke the deployed agent

In [ ]:
from IPython.display import Markdown, display

def ask_agent(prompt):
    """Invoke CertAgent and display the response cleanly."""
    response = agentcore_runtime.invoke({'prompt': prompt})
    text = response.get('response', [''])[0] if isinstance(response.get('response'), list) else response.get('response', '')
    display(Markdown(text))

print('Invoking CertAgent...')
ask_agent('What certificates are expiring soon? Use mock mode and give a prioritized summary in a table.')

In [ ]:
ask_agent('Renew the certificate for api.example.com using mock mode')

In [ ]:
ask_agent('Show the full certificate inventory in a table')

In [ ]:
ask_agent('Scan for certs expiring in 7 days with mock data, then renew any CRITICAL ones')

## Step 11 - Invoke via boto3 (alternative method)

In [ ]:
import boto3, json

agent_arn = launch_result.agent_arn
agentcore_client = boto3.client('bedrock-agentcore', region_name=region)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier='DEFAULT',
    payload=json.dumps({'prompt': 'What is the status of api.example.com?'}),
)

runtime_session_id = boto3_response.get('runtimeSessionId')
print(f'Session ID: {runtime_session_id}')

if 'text/event-stream' in boto3_response.get('contentType', ''):
    for line in boto3_response['response'].iter_lines(chunk_size=1):
        if line:
            line = line.decode('utf-8')
            if line.startswith('data: '):
                print(line[6:])
else:
    events = []
    for event in boto3_response.get('response', []):
        events.append(event)
    if events:
        print(json.loads(events[0].decode('utf-8')))

## Lab 03 Complete

You have:
1. Written a Strands agent with `BedrockAgentCoreApp` (handles HTTP protocol automatically)
2. Deployed to AgentCore Runtime via CodeBuild
3. Invoked the persistent agent with multiple operations
4. Demonstrated boto3 direct invocation

The agent is now running as a managed service.

**Next:** `04_proactive_monitoring.ipynb`